# Práctica 08: Visualización de personas diagnosticadas con diabetes en el estado de Puebla

En esta práctica se realiza el análisis visual de datos relacionados con personas diagnosticadas con diabetes en el estado de Puebla.

### 1. Carga e instalación de librerías

In [1]:
!pip install pandas plotly
!pip install pandas folium geopandas branca -q

Defaulting to user installation because normal site-packages is not writeable
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - ------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "C:\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "C:\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None els

In [2]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster

### 2. Carga de archivos csv

In [10]:
df=pd.read_csv("./dataset_diabetes_puebla.csv")

## 3. Calcular los KPIs

In [11]:
# ---------------------------------------------------------------
# 2. KPI 1-3: VARIABLE OBJETIVO -> estado_glucemico
#    Regla clinica tipo ADA: "el peor criterio gana"
# ---------------------------------------------------------------
def clasificar_glucemia(row):
    es_diabetes = (
        row["glucosa_ayunas_mgdl"] >= 126
        or row["hba1c_pct"] >= 6.5
        or row["ogtt_2h_mgdl"] >= 200
    )
    es_prediabetes = (
        (100 <= row["glucosa_ayunas_mgdl"] < 126)
        or (5.7 <= row["hba1c_pct"] < 6.5)
        or (140 <= row["ogtt_2h_mgdl"] < 200)
    )
    if es_diabetes:
        return "Diabetes"
    elif es_prediabetes:
        return "Prediabetes"
    else:
        return "Normal"
 
df["estado_glucemico"] = df.apply(clasificar_glucemia, axis=1)
 
# ---------------------------------------------------------------
# 3. KPI 4: IMC -> categoria de riesgo
# ---------------------------------------------------------------
def categorizar_imc(imc):
    if imc < 18.5:
        return "Bajo peso"
    elif imc < 25:
        return "Normal"
    elif imc < 30:
        return "Sobrepeso"
    else:
        return "Obesidad (riesgo alto)"
 
df["imc_categoria"] = df["imc"].apply(categorizar_imc)
df["imc_riesgo_alto"] = (df["imc"] >= 30).astype(int)
 
# ---------------------------------------------------------------
# 4. KPI 5: CIRCUNFERENCIA DE CINTURA (depende del genero)
# ---------------------------------------------------------------
def riesgo_cintura(row):
    limite = 102 if row["genero"] == "M" else 88
    return int(row["circunferencia_cintura_cm"] > limite)
 
df["cintura_riesgo_alto"] = df.apply(riesgo_cintura, axis=1)
 
# ---------------------------------------------------------------
# 5. KPI 6: PRESION ARTERIAL -> hipertension
# ---------------------------------------------------------------
df["hipertension"] = (
    (df["presion_sistolica"] >= 130) | (df["presion_diastolica"] >= 80)
).astype(int)
 
# ---------------------------------------------------------------
# 6. KPI 7: PERFIL LIPIDICO -> sindrome metabolico (lipidos)
# ---------------------------------------------------------------
def riesgo_lipidico(row):
    trig_alto = row["trigliceridos_mgdl"] >= 150
    hdl_limite = 40 if row["genero"] == "M" else 50
    hdl_bajo = row["hdl_mgdl"] < hdl_limite
    return int(trig_alto and hdl_bajo)
 
df["riesgo_lipidico"] = df.apply(riesgo_lipidico, axis=1)
 
# ---------------------------------------------------------------
# 7. KPI 8: INSULINA SERICA -> HOMA-IR (resistencia a la insulina)
#    HOMA-IR = (glucosa_ayunas_mg/dL * insulina_uU/mL) / 405
# ---------------------------------------------------------------
df["homa_ir"] = (df["glucosa_ayunas_mgdl"] * df["insulina_serica_uUmL"]) / 405
df["resistencia_insulina"] = (df["homa_ir"] > 2.5).astype(int)
 
# ---------------------------------------------------------------
# 8. KPI 9: EDAD -> factor de riesgo
# ---------------------------------------------------------------
df["edad_riesgo"] = (df["edad"] > 45).astype(int)
 
# ---------------------------------------------------------------
# 9. KPI 10: HISTORIAL FAMILIAR
#    (ya es 0/1 en antecedentes_familiares_diabetes, se deja igual)
# ---------------------------------------------------------------
df["historial_familiar_riesgo"] = df["antecedentes_familiares_diabetes"].astype(int)
 
# ---------------------------------------------------------------
# 10. KPI 12: DIETA -> indice de riesgo dietetico compuesto (0-3)
#     usa las 3 variables de dieta ya presentes en el dataset
# ---------------------------------------------------------------
def riesgo_dieta(row):
    puntos = 0
    if row["indice_calidad_dieta"] < 50:       # escala baja de calidad
        puntos += 1
    if row["azucar_diario_g"] > 50:             # recomendacion OMS azucar libre
        puntos += 1
    if row["frecuencia_ultraprocesados_semana"] > 7:
        puntos += 1
    return puntos
 
df["dieta_riesgo_score"] = df.apply(riesgo_dieta, axis=1)
 
# ---------------------------------------------------------------
# 11. SCORE DE RIESGO GLOBAL (suma de banderas de riesgo modificables)
#     util para practicas de clasificacion supervisada / clustering
# ---------------------------------------------------------------
df["score_riesgo_total"] = (
    df["imc_riesgo_alto"]
    + df["cintura_riesgo_alto"]
    + df["hipertension"]
    + df["riesgo_lipidico"]
    + df["resistencia_insulina"]
    + df["edad_riesgo"]
    + df["historial_familiar_riesgo"]
    + (df["dieta_riesgo_score"] >= 2).astype(int)
)
 
# ---------------------------------------------------------------
# GUARDAR RESULTADO
# ---------------------------------------------------------------
salida = "dataset_diabetes_puebla_kpis.csv"
df.to_csv(salida, index=False)
 
print("KPIs calculados. Columnas nuevas agregadas:")
nuevas = [
    "estado_glucemico", "imc_categoria", "imc_riesgo_alto",
    "cintura_riesgo_alto", "hipertension", "riesgo_lipidico",
    "homa_ir", "resistencia_insulina", "edad_riesgo",
    "historial_familiar_riesgo", "dieta_riesgo_score", "score_riesgo_total"
]
print(nuevas)
print("\nDistribucion de la variable objetivo (estado_glucemico):")
print(df["estado_glucemico"].value_counts())
print(f"\nArchivo guardado en: {salida}")
 

KPIs calculados. Columnas nuevas agregadas:
['estado_glucemico', 'imc_categoria', 'imc_riesgo_alto', 'cintura_riesgo_alto', 'hipertension', 'riesgo_lipidico', 'homa_ir', 'resistencia_insulina', 'edad_riesgo', 'historial_familiar_riesgo', 'dieta_riesgo_score', 'score_riesgo_total']

Distribucion de la variable objetivo (estado_glucemico):
estado_glucemico
Normal         25215
Prediabetes    21867
Diabetes        4918
Name: count, dtype: int64

Archivo guardado en: dataset_diabetes_puebla_kpis.csv
